In [1]:
# merge datasets

import lmdb
import os
from tqdm import tqdm
import pickle
import sys
sys.path.append("/project/Pocket2mol")

def read_lmdb(lmdb_path, mode="direct"):
    """
    Read lmdb file.

    Args:
        lmdb_path (str): Path to the lmdb file.
        mode (str, optional): Read mode. "idx" to follow the idx order, "direct" to read the data directly (use when idx is not continuous).

    Returns:
        list: List of data read from the lmdb file.
    """
    env = lmdb.open(
        lmdb_path,
        subdir=False,
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
        max_readers=256,
    )
    pocket_name_cnt = {}
    smi_cnt={}
    pocket_smi_pair_cnt={}
    txn = env.begin()
    keys = list(txn.cursor().iternext(values=False))
    data_all = []
    if mode == "idx":
        for idx in tqdm(range(len(keys)), desc="read lmdb {}".format(lmdb_path)):
            ky=f'{idx}'.encode()
            datapoint_pickled = txn.get(ky)
            data_piece = pickle.loads(datapoint_pickled)
            data_all.append(data_piece)
            smi=data_piece['smi']
            pocket_name=data_piece['pocket_name']
            pocket_smi_pair=(smi, pocket_name)
            smi_cnt[smi]=smi_cnt.get(smi, 0)+1
            pocket_name_cnt[pocket_name]=pocket_name_cnt.get(pocket_name, 0)+1
            pocket_smi_pair_cnt[pocket_smi_pair]=pocket_smi_pair_cnt.get(pocket_smi_pair, 0)+1
    elif mode == "direct":
        for key in tqdm(keys, desc="read lmdb {}".format(lmdb_path)):
            datapoint_pickled = txn.get(key)
            data_piece = pickle.loads(datapoint_pickled)
            data_all.append((int(key), data_piece))
    return data_all

def write_lmdb(data, lmdb_path):
    env = lmdb.open(
        lmdb_path, 
        subdir=False, 
        readonly=False, 
        lock=False, 
        readahead=False, 
        meminit=False, 
        max_readers=256, 
        map_size=int(10e9)
    )
    with env.begin(write=True, buffers=True) as txn:
        for i, d in tqdm(data):
            txn.put(
                key=f'{i:08d}'.encode(),
                value=pickle.dumps(d)
            )
    env.close()

# scripts of merge lmdbs

DUD_E_lmdb_path = "/data/DTWG_DUD-E/DTWG_DUD-E.lmdb"
# DUD_E_lmdb_path = "/data/lit_pcba/PCBA.lmdb"


input_train_valid_lmdb_path = "/data/pdbbind_cb/bfn_utils/PDBBind_cb_no_test.lmdb"
input_train_valid_index_path = "/data/pdbbind_cb/bfn_utils/id_split_files/PDBBind-DUD_E_FLAPP_0.9_no_test.pt"
output_lmdb_path="/data/pdbbind_cb/bfn_utils/PDBBind_cb.lmdb"
output_id_split_path="/data/pdbbind_cb/bfn_utils/id_split_files/PDBBind-DUD_E_FLAPP_0.9.pt"

# input_train_valid_lmdb_path = "/data/BioLip/bfn_utils/BioLip_no_test.lmdb"
# input_train_valid_index_path = "/data/BioLip/bfn_utils/id_split_files/BioLip-DUD_E_FLAPP_0.9_no_test.pt"
# output_lmdb_path="/data/BioLip/bfn_utils/BioLip.lmdb"
# output_id_split_path="/data/BioLip/bfn_utils/id_split_files/BioLip-DUD_E_FLAPP_0.9.pt"

trian_valid_data=read_lmdb(input_train_valid_lmdb_path)
DUD_E_data=read_lmdb(DUD_E_lmdb_path)

import pickle
import torch

# load index
train_valid_index=torch.load(input_train_valid_index_path)
index=train_valid_index
index['val']=index['test']
index['test']=[]

output_list=trian_valid_data
key_num=len(trian_valid_data)
for data_piece in DUD_E_data:
    output_list.append((key_num, data_piece[1]))
    index['test'].append(key_num)
    key_num+=1


torch.save(index, output_id_split_path)
write_lmdb(output_list, output_lmdb_path)


read lmdb /data/pdbbind_cb/bfn_utils/PDBBind_cb_no_test.lmdb:   0%|          | 0/16260 [00:00<?, ?it/s]

read lmdb /data/pdbbind_cb/bfn_utils/PDBBind_cb_no_test.lmdb: 100%|██████████| 16260/16260 [00:14<00:00, 1119.00it/s]
100%|██████████| 16356/16356 [00:09<00:00, 1702.84it/s]


In [2]:
import lmdb
import os
from tqdm import tqdm
import pickle
import sys
# read lmdb
def read_lmdb(lmdb_path, mode="direct"):
    """
    Read lmdb file.

    Args:
        lmdb_path (str): Path to the lmdb file.
        mode (str, optional): Read mode. "idx" to follow the idx order, "direct" to read the data directly (use when idx is not continuous).

    Returns:
        list: List of data read from the lmdb file.
    """
    env = lmdb.open(
        lmdb_path,
        subdir=False,
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
        max_readers=256,
    )
    pocket_name_cnt = {}
    smi_cnt={}
    pocket_smi_pair_cnt={}
    txn = env.begin()
    keys = list(txn.cursor().iternext(values=False))
    data_all = []
    if mode == "idx":
        for idx in tqdm(range(len(keys)), desc="read lmdb {}".format(lmdb_path)):
            ky=f'{idx}'.encode()
            datapoint_pickled = txn.get(ky)
            data_piece = pickle.loads(datapoint_pickled)
            data_all.append(data_piece)
            smi=data_piece['smi']
            pocket_name=data_piece['pocket_name']
            pocket_smi_pair=(smi, pocket_name)
            smi_cnt[smi]=smi_cnt.get(smi, 0)+1
            pocket_name_cnt[pocket_name]=pocket_name_cnt.get(pocket_name, 0)+1
            pocket_smi_pair_cnt[pocket_smi_pair]=pocket_smi_pair_cnt.get(pocket_smi_pair, 0)+1
    elif mode == "direct":
        for key in tqdm(keys, desc="read lmdb {}".format(lmdb_path)):
            datapoint_pickled = txn.get(key)
            data_piece = pickle.loads(datapoint_pickled)
            data_all.append((int(key), data_piece))
    return data_all



path="/data/DTWG_DUD-E/DTWG_DUD-E.lmdb"
data=read_lmdb(path)

read lmdb /data/DTWG_DUD-E/DTWG_DUD-E.lmdb:   0%|          | 0/101 [00:00<?, ?it/s]

read lmdb /data/DTWG_DUD-E/DTWG_DUD-E.lmdb: 100%|██████████| 101/101 [00:01<00:00, 61.06it/s]


In [6]:
data[0]

(0,
 {'protein_element': tensor([7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8,
          6, 7, 6, 6, 8, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8,
          6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6,
          8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6,
          6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7,
          6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6,
          7, 6, 6, 8, 6, 7, 6, 6, 8, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6,
          7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8,
          6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6,
          8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6,
          6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7,
          6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6, 8, 6, 7, 6, 6,

In [4]:
import torch

file='/data/pdbbind_cb/bfn_utils/id_split_files/PDBBind-DUD_E_FLAPP_0.9_no_test.pt'
data=torch.load(file)
data

{'train': [], 'val': [], 'test': []}

In [1]:
# generate PDBBind_cb.pkl
import sys
sys.path.append("/project/ProFSA")
from scripts.benchmark.dataset import PDBBindCBDataset,DUDECBDataset

dataset=DUDECBDataset()
dataset.generate_targetdiff_index_pkl("/data/DUD-E_cb/DUD-E_cb.pkl")

In [2]:
import pickle

output_id_split_path="/data/DUD-E_cb/DUD-E_cb.pkl"
index=pickle.load(open(output_id_split_path, "rb"))
print(index)
# data_new=read_lmdb(output_lmdb_path)

[('adrb2/5atompoc10A.pdb', 'adrb2/crystal_ligand.mol2'), ('gria2/5atompoc10A.pdb', 'gria2/crystal_ligand.mol2'), ('abl1/5atompoc10A.pdb', 'abl1/crystal_ligand.mol2'), ('parp1/5atompoc10A.pdb', 'parp1/crystal_ligand.mol2'), ('dpp4/5atompoc10A.pdb', 'dpp4/crystal_ligand.mol2'), ('src/5atompoc10A.pdb', 'src/crystal_ligand.mol2'), ('grik1/5atompoc10A.pdb', 'grik1/crystal_ligand.mol2'), ('kpcb/5atompoc10A.pdb', 'kpcb/crystal_ligand.mol2'), ('reni/5atompoc10A.pdb', 'reni/crystal_ligand.mol2'), ('kif11/5atompoc10A.pdb', 'kif11/crystal_ligand.mol2'), ('dyr/5atompoc10A.pdb', 'dyr/crystal_ligand.mol2'), ('hivint/5atompoc10A.pdb', 'hivint/crystal_ligand.mol2'), ('adrb1/5atompoc10A.pdb', 'adrb1/crystal_ligand.mol2'), ('mp2k1/5atompoc10A.pdb', 'mp2k1/crystal_ligand.mol2'), ('aldr/5atompoc10A.pdb', 'aldr/crystal_ligand.mol2'), ('aofb/5atompoc10A.pdb', 'aofb/crystal_ligand.mol2'), ('cp3a4/5atompoc10A.pdb', 'cp3a4/crystal_ligand.mol2'), ('fkb1a/5atompoc10A.pdb', 'fkb1a/crystal_ligand.mol2'), ('fnta/5a

In [13]:
data_new[19398]

(19398,
 (0,
  {'protein_element': tensor([ 7,  6,  6,  8,  6,  6,  6,  7,  6,  7,  7,  1,  1,  1,  1,  1,  1,  7,
            6,  6,  8,  6,  6,  6,  7,  6,  6,  8,  6,  6,  6,  6,  1,  7,  6,  6,
            8,  6,  6,  6,  8,  8,  1,  7,  6,  6,  8,  6,  1,  7,  6,  6,  8,  6,
            6,  6,  6,  1,  7,  6,  6,  8,  6,  6,  6,  6,  1,  7,  6,  6,  8,  6,
            6,  8,  8,  1,  7,  6,  6,  8,  6,  8,  6,  1,  1,  7,  6,  6,  8,  1,
            7,  6,  6,  8,  6,  1,  7,  6,  6,  8,  6,  6,  8,  8,  1,  7,  6,  6,
            8,  6,  6,  8,  8,  1,  7,  6,  6,  8,  6,  8,  6,  1,  1,  7,  6,  6,
            8,  6,  6,  6,  1,  7,  6,  6,  8,  6,  6,  6,  6,  1,  7,  6,  6,  8,
            6,  6,  6,  8,  8,  1,  7,  6,  6,  8,  6,  6,  6,  8,  8,  1,  7,  6,
            6,  8,  6,  6,  6,  6,  7,  1,  7,  6,  6,  8,  6,  6, 16,  6,  1,  7,
            6,  6,  8,  6,  6,  6,  6,  1,  7,  6,  6,  8,  1,  7,  6,  6,  8,  1,
            7,  6,  6,  8,  6,  6,  6,  6,  1,  7,  6, 

In [6]:
import torch

data_file="/mnt/nfs-ssd/data/pdbbind_cb/bfn_utils/PDBBind-DUD_E_FLAPP_0.9_add_aromatic_transformed_simple.pt"

data=torch.load(data_file)

ModuleNotFoundError: No module named 'torch_scatter'

In [6]:
import pexpect

def give_permission(path):

    password = "rootpass"
    command = "chmod -R 777 "+path
    try:
        # Start the su command
        child = pexpect.spawn(f"su")

        # Expect the password prompt
        child.expect("Password:")

        # Send the password
        child.sendline(password)

        # Run the command after logging in
        child.expect(r"#|\$")  # Match the shell prompt
        child.sendline(command)

        # Wait for command execution
        child.expect(r"#|\$")  # Wait for the prompt to return
        output = child.before.decode("utf-8")

        # Exit the shell
        child.sendline("exit")
        child.close()

        return output
    except Exception as e:
        return str(e)

give_permission("/mnt/nfs-ssd/data/itersbdd_bfn/gnina/cm_gnina_0")

' chmod -R 777 /mnt/nfs-ssd/data/itersbdd_bfn/gnina/cm_gnina_0\r0\r\n\x1b]0;root@c0ef713730cc: /\x07root@c0ef713730cc:/'

In [2]:
import torch; print(torch.version.cuda)

12.4


In [2]:
import torch

data = torch.load("/data/pdbbind_2020/bfn_utils/PDBBind-DUD_E_FLAPP_0.9_add_aromatic_transformed_simple.pt")

In [6]:
data["train"]

[ProteinLigandData(protein_pos=[569, 3], protein_atom_feature=[569, 27], protein_element=[569], ligand_pos=[23, 3], ligand_atom_feature_full=[23], ligand_element=[23], protein_filename='5yof/5yof_pocket10A.pdb', ligand_filename='5yof/5yof_ligand.mol2', id=1),
 ProteinLigandData(protein_pos=[480, 3], protein_atom_feature=[480, 27], protein_element=[480], ligand_pos=[11, 3], ligand_atom_feature_full=[11], ligand_element=[11], protein_filename='2hdr/2hdr_pocket10A.pdb', ligand_filename='2hdr/2hdr_ligand.mol2', id=3),
 ProteinLigandData(protein_pos=[1593, 3], protein_atom_feature=[1593, 27], protein_element=[1593], ligand_pos=[40, 3], ligand_atom_feature_full=[40], ligand_element=[40], protein_filename='3eta/3eta_pocket10A.pdb', ligand_filename='3eta/3eta_ligand.mol2', id=4),
 ProteinLigandData(protein_pos=[843, 3], protein_atom_feature=[843, 27], protein_element=[843], ligand_pos=[29, 3], ligand_atom_feature_full=[29], ligand_element=[29], protein_filename='5oq4/5oq4_pocket10A.pdb', ligan